# Evaluation results visualization

**Requires:** `pip install pandas matplotlib`

- Group by `story_prompt`; average multiple rows per prompt before macro-averages across prompts.
- For flag ablations, fix `specialist_model_variant_tag` (e.g. `baseline_tango2`) so specialist swaps do not confound flag effects.

In [5]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Literal, Mapping, Optional, Tuple

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path


## Constants

In [6]:
BOOL_FLAG_COLUMNS: Tuple[str, ...] = (
    "flag_fill_coverage_by_llm",
    "flag_use_dsp",
    "flag_use_movie_bgms",
    "flag_use_narrator",
    "flag_use_llm_to_predict_align",
    "flag_use_dsp_to_predict_align",
    "flag_use_avg_llm_and_dsp_to_predict_align",
    "flag_use_dl_based_llm_and_dsp_alignment_predictor",
)

DEFAULT_OBJECTIVE: Dict[str, Literal["maximize", "minimize"]] = {
    "clap_score": "maximize",
    "audio_richness_spectral_flatness": "maximize",
    "audio_richness_spectral_entropy": "maximize",
    "noise_floor_db": "maximize",
    "audio_onsets": "maximize",
    "yt_coverage_score": "maximize",
    "yt_sync_score": "maximize",
    "yt_coverage_and_sync_coverage_score": "maximize",
    "yt_coverage_and_sync_sync_score": "maximize",
    "cinematic_dynamic_range_db": "maximize",
    "cinematic_crest_factor": "maximize",
    "cinematic_spectral_flatness": "maximize",
    "cinematic_spectral_entropy": "maximize",
    "cinematic_spectral_centroid_hz": "maximize",
    "spectral_kl_divergence": "minimize",
    "fad_score": "minimize",
    "total_seconds": "minimize",
    "pipeline_total_seconds": "minimize",
    "evaluation_seconds": "minimize",
}

## Load and clean CSV

In [7]:
def _parse_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    up = s.astype(str).str.strip().str.upper()
    return up.map({"TRUE": True, "FALSE": False, "1": True, "0": False}).fillna(False)


def load_results_csv(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)
    for col in BOOL_FLAG_COLUMNS:
        if col in df.columns:
            df[col] = _parse_bool_series(df[col])
    return df


def clean_results_df(
    df: pd.DataFrame,
    *,
    drop_empty_experiment_tag: bool = True,
    exclude_errors: bool = True,
) -> pd.DataFrame:
    out = df.copy()
    if drop_empty_experiment_tag and "experiment_tag" in out.columns:
        out = out[out["experiment_tag"].notna() & (out["experiment_tag"].astype(str).str.strip() != "")]
    if exclude_errors and "error" in out.columns:
        err = out["error"]
        mask = err.isna() | (err.astype(str).str.strip() == "")
        out = out[mask]
    return out

## Configuration: infer, filter, objective


In [8]:
def infer_baseline_config(df: pd.DataFrame) -> Dict[str, Any]:
    """Defaults from baseline experiment + baseline_tango2 specialist."""
    sub = df[
        (df["experiment_tag"].astype(str) == "baseline")
        & (df["specialist_model_variant_tag"].astype(str) == "baseline_tango2")
    ]
    if sub.empty:
        sub = df[df["experiment_tag"].astype(str) == "baseline"]
    if sub.empty:
        raise ValueError("No baseline rows found to infer default configuration.")
    row = sub.iloc[0]
    cfg: Dict[str, Any] = {}
    for c in BOOL_FLAG_COLUMNS:
        if c in row.index:
            cfg[c] = bool(row[c])
    for c in ("decide_audio_model_name", "specialist_model_variant_tag"):
        if c in row.index and pd.notna(row[c]):
            cfg[c] = str(row[c])
    return cfg


def filter_by_configuration(df: pd.DataFrame, config: Mapping[str, Any]) -> pd.DataFrame:
    out = df
    for key, val in config.items():
        if key not in out.columns:
            raise KeyError(f"Unknown column in config: {key}")
        if key in BOOL_FLAG_COLUMNS:
            out = out[out[key] == bool(val)]
        else:
            out = out[out[key].astype(str) == str(val)]
    return out


def resolve_objective(
    metric_col: str, objective: Optional[Literal["maximize", "minimize"]]
) -> Literal["maximize", "minimize"]:
    if objective is not None:
        return objective
    if metric_col in DEFAULT_OBJECTIVE:
        return DEFAULT_OBJECTIVE[metric_col]
    return "maximize"


In [9]:
@dataclass
class ConfigurationPlotInsights:
    metric_col: str
    n_prompts: int
    mean_over_prompts: float
    best_prompt: str
    best_value: float
    worst_prompt: str
    worst_value: float
    objective: str


## Plot: one configuration (per prompt)


In [10]:
def plot_metric_for_configuration(
    df: pd.DataFrame,
    metric_col: str,
    configuration: Optional[Mapping[str, Any]] = None,
    *,
    output_path: str,
    aggregate_how: Literal["mean"] = "mean",
    objective: Optional[Literal["maximize", "minimize"]] = None,
    exclude_errors: bool = True,
    prompt_label_max_len: int = 56,
    dpi: int = 150,
    save_pdf: bool = False,
) -> Tuple[pd.DataFrame, ConfigurationPlotInsights]:
    """
    Filter rows to ``configuration`` (defaults merged from infer_baseline_config),
    aggregate metric per story_prompt, plot horizontal bars, save figure.
    Returns (per_prompt DataFrame, insights).
    """
    obj = resolve_objective(metric_col, objective)
    work = clean_results_df(df, exclude_errors=exclude_errors) if exclude_errors else df.copy()

    cfg: Dict[str, Any] = dict(infer_baseline_config(work))
    if configuration:
        cfg.update(dict(configuration))

    filt = filter_by_configuration(work, cfg)
    if metric_col not in filt.columns:
        raise KeyError(f"Unknown metric column: {metric_col}")

    filt = filt.assign(**{metric_col: pd.to_numeric(filt[metric_col], errors="coerce")})
    filt = filt[filt[metric_col].notna()]
    if filt.empty:
        raise ValueError("No rows left after filtering for configuration and valid metric.")

    if aggregate_how == "mean":
        g = filt.groupby("story_prompt", as_index=False)[metric_col].mean()
    else:
        raise ValueError(f"Unsupported aggregate_how: {aggregate_how}")

    g = g.sort_values(metric_col, ascending=(obj == "minimize"))
    prompts = g["story_prompt"].astype(str).tolist()
    values = g[metric_col].astype(float).tolist()

    def short_label(p: str, i: int) -> str:
        s = p if len(p) <= prompt_label_max_len else p[: prompt_label_max_len - 1] + "…"
        return f"{i + 1}. {s}"

    labels = [short_label(p, i) for i, p in enumerate(prompts)]
    fig_h = max(6.0, 0.22 * len(labels))
    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.barh(range(len(labels)), values, color="steelblue")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(metric_col)
    ax.set_title(f"{metric_col} per story_prompt\nconfig={cfg}")
    ax.invert_yaxis()
    fig.tight_layout()
    d = os.path.dirname(os.path.abspath(output_path))
    if d:
        os.makedirs(d, exist_ok=True)
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    if save_pdf:
        fig.savefig(os.path.splitext(output_path)[0] + ".pdf", bbox_inches="tight")
    plt.close(fig)

    mean_v = float(sum(values) / len(values)) if values else float("nan")
    if obj == "maximize":
        bi = max(range(len(values)), key=lambda i: values[i])
        wi = min(range(len(values)), key=lambda i: values[i])
    else:
        bi = min(range(len(values)), key=lambda i: values[i])
        wi = max(range(len(values)), key=lambda i: values[i])

    insights = ConfigurationPlotInsights(
        metric_col=metric_col,
        n_prompts=len(values),
        mean_over_prompts=mean_v,
        best_prompt=prompts[bi],
        best_value=float(values[bi]),
        worst_prompt=prompts[wi],
        worst_value=float(values[wi]),
        objective=obj,
    )
    print(
        f"[{metric_col}] mean_over_prompts={insights.mean_over_prompts:.6g} "
        f"best_prompt (truncated)={insights.best_prompt[:prompt_label_max_len]!r} "
        f"best_value={insights.best_value:.6g} objective={obj}"
    )
    return g.rename(columns={metric_col: f"mean_{metric_col}"}), insights


## Helpers for ablation comparison


In [11]:

def _per_prompt_means_for_slice(
    slice_df: pd.DataFrame,
    metric_col: str,
) -> pd.Series:
    s = pd.to_numeric(slice_df[metric_col], errors="coerce")
    tmp = slice_df.assign(_m=s)
    tmp = tmp[tmp["_m"].notna()]
    if tmp.empty:
        return pd.Series(dtype=float)
    return tmp.groupby("story_prompt")["_m"].mean()

In [12]:
def _ordered_experiment_tags(tags: Iterable[str]) -> List[str]:
    seen = set()
    ordered: List[str] = []
    for t in tags:
        if t and t not in seen:
            seen.add(t)
            ordered.append(t)

    def sort_key(t: str) -> Tuple[int, str]:
        if t == "baseline":
            return (0, t)
        if t.startswith("decide_model_"):
            return (1, t)
        if t.startswith("toggle_"):
            return (2, t)
        return (3, t)

    return sorted(ordered, key=sort_key)


In [13]:
@dataclass
class AblationRow:
    label: str
    mean_over_prompts: float
    best_single_prompt_value: float
    n_prompts: int


In [14]:
def compare_ablations_for_metric(
    df: pd.DataFrame,
    metric_col: str,
    *,
    objective: Optional[Literal["maximize", "minimize"]] = None,
    specialist_model_variant_tag: str = "baseline_tango2",
    output_dir: str,
    include_decide_model: bool = True,
    include_specialist_ablations: bool = False,
    exclude_errors: bool = True,
    dpi: int = 150,
) -> Tuple[pd.DataFrame, str]:
    """
    For each experiment_tag (optionally plus specialist-only bars), compute macro mean
    across prompts and best single-prompt mean; save two bar plots and a text report.
    Returns (summary DataFrame, report text).
    """
    obj = resolve_objective(metric_col, objective)
    work = clean_results_df(df, exclude_errors=exclude_errors) if exclude_errors else df.copy()
    if metric_col not in work.columns:
        raise KeyError(f"Unknown metric column: {metric_col}")

    os.makedirs(output_dir, exist_ok=True)

    rows_out: List[AblationRow] = []

    tag_series = work["experiment_tag"].astype(str)
    all_tags = _ordered_experiment_tags(tag_series.unique())

    def process_tag(ex_tag: str, label: Optional[str] = None) -> None:
        lab = label or ex_tag
        sub = work[
            (tag_series == ex_tag) & (work["specialist_model_variant_tag"].astype(str) == specialist_model_variant_tag)
        ]
        pm = _per_prompt_means_for_slice(sub, metric_col)
        if pm.empty:
            rows_out.append(
                AblationRow(label=lab, mean_over_prompts=float("nan"), best_single_prompt_value=float("nan"), n_prompts=0)
            )
            return
        mean_v = float(pm.mean())
        if obj == "maximize":
            best_v = float(pm.max())
        else:
            best_v = float(pm.min())
        rows_out.append(
            AblationRow(label=lab, mean_over_prompts=mean_v, best_single_prompt_value=best_v, n_prompts=int(pm.shape[0]))
        )

    for ex_tag in all_tags:
        if ex_tag == "baseline":
            process_tag("baseline")
            continue
        if ex_tag.startswith("decide_model_"):
            if include_decide_model:
                process_tag(ex_tag)
            continue
        if ex_tag.startswith("toggle_"):
            process_tag(ex_tag)
            continue
        process_tag(ex_tag)

    if include_specialist_ablations:
        baseline_only = work[tag_series == "baseline"]
        spec_tags = sorted(
            baseline_only["specialist_model_variant_tag"].dropna().astype(str).unique(),
            key=lambda x: (0 if x == "baseline_tango2" else 1, x),
        )
        for st in spec_tags:
            if st == specialist_model_variant_tag:
                continue
            sub = baseline_only[baseline_only["specialist_model_variant_tag"].astype(str) == st]
            pm = _per_prompt_means_for_slice(sub, metric_col)
            lab = f"baseline | specialist={st}"
            if pm.empty:
                rows_out.append(
                    AblationRow(label=lab, mean_over_prompts=float("nan"), best_single_prompt_value=float("nan"), n_prompts=0)
                )
            else:
                mean_v = float(pm.mean())
                best_v = float(pm.max()) if obj == "maximize" else float(pm.min())
                rows_out.append(
                    AblationRow(
                        label=lab, mean_over_prompts=mean_v, best_single_prompt_value=best_v, n_prompts=int(pm.shape[0])
                    )
                )

    summary = pd.DataFrame([r.__dict__ for r in rows_out])

    def _plot_bars(column: str, title: str, fname: str) -> None:
        sub2 = summary.dropna(subset=[column]).copy()
        if sub2.empty:
            return
        sub2 = sub2.sort_values(column, ascending=(obj == "minimize"))
        labels = sub2["label"].tolist()
        vals = sub2[column].tolist()
        fig_w = max(10.0, 0.35 * len(labels))
        fig, ax = plt.subplots(figsize=(fig_w, 6))
        color = "seagreen" if column == "mean_over_prompts" else "darkorange"
        ax.bar(range(len(labels)), vals, color=color)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=8)
        ax.set_ylabel(column)
        ax.set_title(title)
        fig.tight_layout()
        path = os.path.join(output_dir, fname)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)

    _plot_bars("mean_over_prompts", f"Mean over prompts — {metric_col}", f"ablations_mean_{metric_col}.png")
    _plot_bars("best_single_prompt_value", f"Best single-prompt mean — {metric_col}", f"ablations_best_{metric_col}.png")

    valid_mean = summary.dropna(subset=["mean_over_prompts"])
    valid_best = summary.dropna(subset=["best_single_prompt_value"])

    report_lines: List[str] = [
        f"# Ablation comparison: {metric_col}",
        f"objective: {obj}",
        f"specialist_model_variant_tag filter: {specialist_model_variant_tag!r}",
        "",
    ]

    if not valid_mean.empty:
        if obj == "maximize":
            win_mean = valid_mean.loc[valid_mean["mean_over_prompts"].idxmax()]
        else:
            win_mean = valid_mean.loc[valid_mean["mean_over_prompts"].idxmin()]
        report_lines.append(f"## Best configuration by mean over prompts: {win_mean['label']}")
        report_lines.append(f"value: {win_mean['mean_over_prompts']:.6g} (n_prompts={win_mean['n_prompts']})")
        report_lines.append("")
    if not valid_best.empty:
        if obj == "maximize":
            win_best = valid_best.loc[valid_best["best_single_prompt_value"].idxmax()]
        else:
            win_best = valid_best.loc[valid_best["best_single_prompt_value"].idxmin()]
        report_lines.append(f"## Best configuration by peak single-prompt score: {win_best['label']}")
        report_lines.append(f"value: {win_best['best_single_prompt_value']:.6g} (n_prompts={win_best['n_prompts']})")
        report_lines.append("")

    report_lines.append("## Table")
    report_lines.append(summary.to_string(index=False))

    if not valid_mean.empty:
        baseline_row = valid_mean[valid_mean["label"] == "baseline"]
        if not baseline_row.empty:
            b = float(baseline_row.iloc[0]["mean_over_prompts"])
            others = valid_mean[valid_mean["label"] != "baseline"].copy()
            if not others.empty:
                others = others.copy()
                others["delta_vs_baseline"] = others["mean_over_prompts"] - b
                if obj == "maximize":
                    worst = others.loc[others["delta_vs_baseline"].idxmin()]
                    report_lines.append(
                        f"\n## Largest mean drop vs baseline\n{worst['label']} (delta={worst['delta_vs_baseline']:.6g})"
                    )
                else:
                    worst = others.loc[others["delta_vs_baseline"].idxmax()]
                    report_lines.append(
                        f"\n## Largest mean increase vs baseline (worse)\n{worst['label']} (delta={worst['delta_vs_baseline']:.6g})"
                    )

    report_text = "\n".join(report_lines)
    rep_path = os.path.join(output_dir, f"ablation_report_{metric_col}.txt")
    with open(rep_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print(f"Wrote report: {rep_path}")

    return summary, report_text



## Per-prompt consensus (matrix, wins, heatmap)

- **Intersection only**: prompts with a finite score for every ablation column.
- **Fractional wins**: k-way ties split credit as 1/k per ablation.
- Use `compare_ablations_for_metric_extended` in the Run cell for CSV + PNGs under `Results/plots/<metric>/`.


In [ ]:
def _sanitize_metric_for_path(metric_col: str) -> str:
    return metric_col.replace(os.sep, "_").replace("/", "_")


def _iter_ablation_per_prompt_series(
    work: pd.DataFrame,
    metric_col: str,
    *,
    specialist_model_variant_tag: str,
    include_decide_model: bool,
    include_specialist_ablations: bool,
) -> Iterable[Tuple[str, pd.Series]]:
    """
    Yields (display_label, Series indexed by story_prompt with mean metric per prompt)
    in the same order as compare_ablations_for_metric.
    """
    tag_series = work["experiment_tag"].astype(str)
    all_tags = _ordered_experiment_tags(tag_series.unique())

    def one(ex_tag: str, label: Optional[str] = None) -> Optional[Tuple[str, pd.Series]]:
        lab = label or ex_tag
        sub = work[
            (tag_series == ex_tag) & (work["specialist_model_variant_tag"].astype(str) == specialist_model_variant_tag)
        ]
        pm = _per_prompt_means_for_slice(sub, metric_col)
        if pm.empty:
            return None
        return (lab, pm)

    for ex_tag in all_tags:
        if ex_tag == "baseline":
            out = one("baseline")
            if out:
                yield out
            continue
        if ex_tag.startswith("decide_model_"):
            if include_decide_model:
                out = one(ex_tag)
                if out:
                    yield out
            continue
        if ex_tag.startswith("toggle_"):
            out = one(ex_tag)
            if out:
                yield out
            continue
        out = one(ex_tag)
        if out:
            yield out

    if include_specialist_ablations:
        baseline_only = work[tag_series == "baseline"]
        spec_tags = sorted(
            baseline_only["specialist_model_variant_tag"].dropna().astype(str).unique(),
            key=lambda x: (0 if x == "baseline_tango2" else 1, x),
        )
        for st in spec_tags:
            if st == specialist_model_variant_tag:
                continue
            sub = baseline_only[baseline_only["specialist_model_variant_tag"].astype(str) == st]
            pm = _per_prompt_means_for_slice(sub, metric_col)
            lab = f"baseline | specialist={st}"
            if not pm.empty:
                yield (lab, pm)


def ablation_per_prompt_matrix(
    df: pd.DataFrame,
    metric_col: str,
    *,
    objective: Optional[Literal["maximize", "minimize"]] = None,
    specialist_model_variant_tag: str = "baseline_tango2",
    include_decide_model: bool = True,
    include_specialist_ablations: bool = False,
    exclude_errors: bool = True,
) -> pd.DataFrame:
    """
    Build a matrix: rows = story_prompt, columns = ablation labels, values = mean metric.
    Keeps only prompts that have a finite value for every ablation column (intersection).
    """
    resolve_objective(metric_col, objective)
    work = clean_results_df(df, exclude_errors=exclude_errors) if exclude_errors else df.copy()
    if metric_col not in work.columns:
        raise KeyError(f"Unknown metric column: {metric_col}")

    pieces: Dict[str, pd.Series] = {}
    for lab, ser in _iter_ablation_per_prompt_series(
        work,
        metric_col,
        specialist_model_variant_tag=specialist_model_variant_tag,
        include_decide_model=include_decide_model,
        include_specialist_ablations=include_specialist_ablations,
    ):
        if lab in pieces:
            continue
        pieces[lab] = ser

    if not pieces:
        return pd.DataFrame()

    mat = pd.concat(pieces, axis=1)
    mat.columns = [str(c) for c in mat.columns]
    return mat.dropna(how="any")


def summarize_prompt_wins(
    matrix: pd.DataFrame,
    objective: Literal["maximize", "minimize"],
    *,
    baseline_label: str = "baseline",
) -> pd.DataFrame:
    """
    Per ablation: strict wins (sole best on a prompt), fractional wins (1/k on k-way ties),
    win rates, and beats_baseline count if baseline column exists.
    Ties: each tied ablation receives 1/k of a win for that prompt.
    """
    if matrix.empty or matrix.shape[1] < 1:
        return pd.DataFrame()

    ascending = objective == "minimize"
    n = len(matrix)

    strict = {c: 0 for c in matrix.columns}
    fractional = {c: 0.0 for c in matrix.columns}

    for _, row in matrix.iterrows():
        vals = row.to_numpy(dtype=float)
        if ascending:
            best = np.nanmin(vals)
            is_best = np.isclose(vals, best, rtol=1e-9, atol=1e-12)
        else:
            best = np.nanmax(vals)
            is_best = np.isclose(vals, best, rtol=1e-9, atol=1e-12)
        k = int(is_best.sum())
        if k == 0:
            continue
        share = 1.0 / k
        for j, col in enumerate(matrix.columns):
            if is_best[j]:
                fractional[col] += share
                if k == 1:
                    strict[col] += 1

    rows = []
    for col in matrix.columns:
        rows.append(
            {
                "ablation": col,
                "prompts_won_strict": strict[col],
                "prompts_won_fractional": fractional[col],
                "win_rate_strict": strict[col] / n if n else float("nan"),
                "win_rate_fractional": fractional[col] / n if n else float("nan"),
            }
        )
    out = pd.DataFrame(rows)

    if baseline_label in matrix.columns:
        b = matrix[baseline_label].to_numpy(dtype=float)
        beats = []
        for col in matrix.columns:
            if col == baseline_label:
                beats.append(float("nan"))
                continue
            v = matrix[col].to_numpy(dtype=float)
            if ascending:
                beats.append(float(np.sum(v < b)))
            else:
                beats.append(float(np.sum(v > b)))
        out["beats_baseline_count"] = beats
        out["beats_baseline_rate"] = out["beats_baseline_count"] / n if n else float("nan")

    return out.sort_values("win_rate_fractional", ascending=False, ignore_index=True)


def mean_rank_per_ablation(
    matrix: pd.DataFrame,
    objective: Literal["maximize", "minimize"],
) -> pd.DataFrame:
    """Mean rank per column (1 = best). Ties get the average rank."""
    if matrix.empty:
        return pd.DataFrame()
    ascending = objective == "minimize"
    r = matrix.rank(axis=1, ascending=ascending, method="average")
    out = r.mean(axis=0).reset_index()
    out.columns = ["ablation", "mean_rank"]
    return out.sort_values("mean_rank", ascending=True, ignore_index=True)



In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

METRIC = "clap_score"
SPECIALIST_TAG = "baseline_tango2"
CSV_PATH = NOTEBOOK_DIR / "Results" / "final_results.csv"
PLOTS_ROOT = NOTEBOOK_DIR / "Results" / "plots"
INCLUDE_DECIDE_MODEL = True
INCLUDE_SPECIALIST_ABLATIONS = False
OBJECTIVE = None

df_raw = load_results_csv(str(CSV_PATH))
df_clean = clean_results_df(df_raw)
print(len(df_clean), "rows after clean")

summary, report_text, matrix, wins_df, ranks_df = compare_ablations_for_metric_extended(
    df_clean,
    METRIC,
    objective=OBJECTIVE,
    specialist_model_variant_tag=SPECIALIST_TAG,
    output_dir=str(PLOTS_ROOT),
    include_decide_model=INCLUDE_DECIDE_MODEL,
    include_specialist_ablations=INCLUDE_SPECIALIST_ABLATIONS,
)
display(summary)
if not matrix.empty:
    display(wins_df.head(15))
    display(ranks_df.head(15))

run_sub = NOTEBOOK_DIR / "Results" / "plots" / _sanitize_metric_for_path(METRIC)
per_prompt_df, insights = plot_metric_for_configuration(
    df_clean,
    METRIC,
    configuration=None,
    output_path=str(run_sub / f"per_prompt_{METRIC}_baseline.png"),
    objective=OBJECTIVE,
)
display(per_prompt_df.head(10))
insights


## Run: load CSV, ablation comparison, baseline per-prompt plot


In [25]:
try:
    from IPython.display import display
except ImportError:
    display = print

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

METRIC = "clap_score"
SPECIALIST_TAG = "baseline_tango2"
CSV_PATH = NOTEBOOK_DIR / "Results" / "final_results.csv"
PLOTS_ROOT = NOTEBOOK_DIR / "Results" / "plots"
INCLUDE_DECIDE_MODEL = True
INCLUDE_SPECIALIST_ABLATIONS = False
OBJECTIVE = None

df_raw = load_results_csv(str(CSV_PATH))
df_clean = clean_results_df(df_raw)
print(len(df_clean), "rows after clean")

summary, report_text, matrix, wins_df, ranks_df = compare_ablations_for_metric_extended(
    df_clean,
    METRIC,
    objective=OBJECTIVE,
    specialist_model_variant_tag=SPECIALIST_TAG,
    output_dir=str(PLOTS_ROOT),
    include_decide_model=INCLUDE_DECIDE_MODEL,
    include_specialist_ablations=INCLUDE_SPECIALIST_ABLATIONS,
)
display(summary)
if not matrix.empty:
    display(wins_df.head(15))
    display(ranks_df.head(15))

run_sub = NOTEBOOK_DIR / "Results" / "plots" / _sanitize_metric_for_path(METRIC)
per_prompt_df, insights = plot_metric_for_configuration(
    df_clean,
    METRIC,
    configuration=None,
    output_path=str(run_sub / f"per_prompt_{METRIC}_baseline.png"),
    objective=OBJECTIVE,
)
display(per_prompt_df.head(10))
insights


1288 rows after clean
Wrote report: /Users/ajitesh/Desktop/BTP/cinemaudio-studio/backend/Evaluation/Results/plots/audio_richness_spectral_flatness/ablation_report_audio_richness_spectral_flatness.txt


,label,mean_over_prompts,best_single_prompt_value,n_prompts
0,baseline,0.037154,0.424969,39
1,decide_model_gemini-2.5-flash,0.026091,0.132501,36
2,toggle_fill_coverage_by_llm,0.021126,0.083243,35
3,toggle_use_avg_llm_and_dsp_to_predict_align,0.023885,0.158207,29
4,toggle_use_dl_based_llm_and_dsp_alignment_pred...,0.021619,0.110239,29
5,toggle_use_dsp,0.034076,0.212225,35
6,toggle_use_dsp_to_predict_align,0.028041,0.176214,30
7,toggle_use_llm_to_predict_align,0.057154,0.900850,30
8,toggle_use_movie_bgms,0.024297,0.239298,32
9,toggle_use_narrator,0.025663,0.151053,31


[audio_richness_spectral_flatness] mean_over_prompts=0.0375508 best_prompt (truncated)='An armored, monstrous creature grunts and struggles to p' best_value=0.424969 objective=maximize


,story_prompt,mean_audio_richness_spectral_flatness
21,"An armored, monstrous creature grunts and stru...",0.424969
32,The final moments of the trailer reveal the fi...,0.267122
34,The scene transitions from the worried faces o...,0.095020
33,"The scene abruptly shifts from a dimly lit, in...",0.094597
37,Within the grim confines of a prison yard and ...,0.044337
22,An astronaut frantically grapples with failing...,0.040262
14,A young girl transitions from an innocent conv...,0.035934
13,"A whirlwind montage of lavish excess, power-fl...",0.026440
9,"A series of tense, rain-soaked western confron...",0.024285
17,Amidst a crumbling future where humanity immer...,0.023385


ConfigurationPlotInsights(metric_col='audio_richness_spectral_flatness', n_prompts=38, mean_over_prompts=0.03755084125526313, best_prompt='An armored, monstrous creature grunts and struggles to pull itself through a dense, muddy forest floor, punctuated by on-screen text, before a dramatic transition to the iconic studio logo and a final ominous silhouette of a warrior.', best_value=0.4249688, worst_prompt='As an ancient ritual inadvertently unleashes terrifying demonic forces upon a modern city, desperate citizens and heroes alike must confront overwhelming destruction and grotesque horrors, culminating in a poignant moment of familial fear amidst the chaos.', worst_value=0.0059009814, objective='maximize')